# Embryo AI: Clinical Master (v4.0 - EfficientNet Architecture)

### 🚀 Version 4 Strategy:
- **Backbone**: EfficientNet-B0 (1280 features).
- **Loss Strategy**: Hybrid (Standard EXP, Weighted Focal ICM/TE).
- **Head Weights**: Prioritizing secondary heads (ICM=2.0x, TE=2.0x).
- **Optimization**: ReduceLROnPlateau Scheduler.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import zipfile
import time
from sklearn.utils import class_weight

# --- CONFIG ---
CONFIG = {
    "BATCH_SIZE": 16,
    "EPOCHS": 15,
    "LEARNING_RATE": 0.0005,
    "DATA_DIR": "blasto2k_dataset/Images",
    "TRAIN_CSV": "blasto2k_dataset/Gardner_train_silver.csv",
    "TEST_CSV": "blasto2k_dataset/Gardner_test_gold.csv"
}
if not os.path.exists(CONFIG["TEST_CSV"]): 
    CONFIG["TEST_CSV"] = "blasto2k_dataset/Gardner_test_gold_onlyGardnerScores.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Professional Architecture (v4)

In [ ]:
class MultiHeadEfficientNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = models.efficientnet_b0(weights='DEFAULT')
        # The classifier of EffNet-B0 has 1280 input features
        self.base.classifier = nn.Identity()
        
        self.h_exp = nn.Linear(1280, 5)
        self.h_icm = nn.Linear(1280, 4)
        self.h_te = nn.Linear(1280, 4)
        
    def forward(self, x):
        f = torch.flatten(self.base(x), 1)
        return self.h_exp(f), self.h_icm(f), self.h_te(f)

class HybridLoss(nn.Module):
    def __init__(self, w_icm, w_te, gamma=2):
        super().__init__()
        self.gamma = gamma
        self.w_icm = w_icm # Tensor weights for ICM classes
        self.w_te = w_te   # Tensor weights for TE classes
        
    def focal_loss(self, inputs, targets, weights):
        ce = F.cross_entropy(inputs, targets, weight=weights, reduction='none')
        pt = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()

    def forward(self, p_exp, t_exp, p_icm, t_icm, p_te, t_te):
        # Standard CE for Expansion (1.0x weight)
        l_exp = F.cross_entropy(p_exp, t_exp)
        # Weighted Focal for ICM and TE (2.0x head weight)
        l_icm = self.focal_loss(p_icm, t_icm, self.w_icm)
        l_te = self.focal_loss(p_te, t_te, self.w_te)
        
        return 1.0 * l_exp + 2.0 * l_icm + 2.0 * l_te

## 2. Advanced Training Pipeline

In [ ]:
trans = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_set = BlastoDataset(CONFIG['TRAIN_CSV'], CONFIG['DATA_DIR'], trans)
test_set = BlastoDataset(CONFIG['TEST_CSV'], CONFIG['DATA_DIR'], trans)

train_loader = DataLoader(train_set, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
test_loader = DataLoader(test_set, batch_size=CONFIG['BATCH_SIZE'])

# Calculate weights for ICM/TE
def get_weights(column):
    vals = train_set.df[column].apply(lambda x: train_set._map_label(x))
    w = class_weight.compute_class_weight('balanced', classes=np.unique(vals), y=vals)
    return torch.tensor(w, dtype=torch.float32).to(device)

w_icm, w_te = get_weights(train_set.icm_col), get_weights(train_set.te_col)

model = MultiHeadEfficientNet().to(device)
criterion = HybridLoss(w_icm, w_te)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['LEARNING_RATE'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

print(f"🚀 V4 START: EfficientNet-B0 Hybrid Training ({len(train_set)} samples)")
for ep in range(CONFIG['EPOCHS']):
    model.train(); tl = 0
    for imgs, exps, icms, tes in train_loader:
        imgs, exps, icms, tes = imgs.to(device), exps.to(device), icms.to(device), tes.to(device)
        optimizer.zero_grad()
        pe, pi, pt = model(imgs)
        loss = criterion(pe, exps, pi, icms, pt, tes)
        loss.backward(); optimizer.step(); tl += loss.item()
    
    avg_loss = tl/len(train_loader)
    scheduler.step(avg_loss)
    print(f"Epoch {ep+1} | Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

torch.save(model.state_dict(), "embryo_grading_v4.pth")